# 🏥 GIEndoDL: TASK D46 - HUẤN LUYỆN THỬ NGHIỆM (DRY-RUN) 10 EPOCHS
## Đề tài: Ứng dụng Học Sâu trong Nhận diện và Phân loại Tổn thương Đường Tiêu hóa từ Ảnh Nội soi
**Học viên thực hiện:** Lê Huy Phát | **Đơn vị:** Đại học Công Thương TP.HCM (HUIT)  
**Kiến trúc:** Hybrid CNN-CBAM-Transformer (ResNet-50 + CBAM Stage 4 + 2-layer Transformer + GAP Head)  
**Chiến lược Dữ liệu (DCAI):** Schema 17 phân lớp y sinh sạch (Zero Video Leakage, Aspect-Ratio Letterbox, Class-Balanced Focal Loss)  
**Mục tiêu Task D46:** Kiểm tra tốc độ hội tụ mô hình và thông lượng DataLoader trên Kaggle GPU (Tesla P100 / T4) trước khi mở rộng Full Run.

---
### 📋 Tóm tắt Cấu hình Tối ưu:
1. **Phân lớp:** 17 Classes chuẩn hóa y sinh (Imbalance Ratio giảm từ $191.5:1$ xuống $12.63:1$, loại bỏ hoàn toàn các lớp $< 91$ mẫu).
2. **Phân hoạch:** GroupStratifiedKFold (70% Train: 7,452 | 15% Val: 1,601) đảm bảo rò rỉ bệnh nhân/video $\Lambda = 0.00000000\%$.
3. **Tiền xử lý:** Cắt viền đen Otsu + Che text overlay + Giảm phản quang mềm + CLAHE kênh L + Aspect-Ratio Letterbox (0% méo).
4. **Tăng cường y sinh:** Phản chiếu, quay $90^\circ$, biến dạng vòm kính, làm mờ nhu động ruột (Tuyệt đối không dùng CutMix/MixUp).
5. **Hàm mất mát:** Class-Balanced Focal Loss ($\beta = 0.75, \gamma = 2.0$, Label Smoothing $0.05$).
6. **Bộ lấy mẫu:** WeightedRandomSampler với tham số lũy thừa $\beta = 0.75$ san phẳng phân bố mini-batch.
7. **Tối ưu hóa:** AdamW (lr=$10^{-4}$, weight_decay=$10^{-2}$), CosineAnnealingLR (10 epochs), AMP FP16 Mixed Precision.

In [ ]:
# 1. KIỂM TRA PHẦN CỨNG GPU & MÔI TRƯỜNG KAGGLE
!nvidia-smi

import os
import sys
import time
import math
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 80)
print(f'🚀 PyTorch Version: {torch.__version__}')
print(f'⚡ Device: {device} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🎮 GPU Model: {torch.cuda.get_device_name(0)}')
    print(f'💾 Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
print('=' * 80)

In [ ]:
# 2. CÀI ĐẶT CÁC THƯ VIỆN BỔ TRỢ
!pip install -q albumentations timm scikit-learn matplotlib seaborn tqdm

In [ ]:
# 3. THIẾT LẬP REPO HOẶC NẠP DỮ LIỆU TỰ ĐỘNG
import subprocess

# Kiểm tra nếu đang chạy trong Kaggle và chưa clone repo
REPO_DIR = Path("/kaggle/working/GIEndoDL")
if not REPO_DIR.exists():
    print("⚡ Đang clone mã nguồn GIEndoDL từ GitHub...")
    subprocess.run(["git", "clone", "https://github.com/LeHuyPhat2907/GIEndoDL.git", str(REPO_DIR)], check=False)

if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
    print(f"✅ Đã liên kết đường dẫn repo: {REPO_DIR}")
else:
    # Đường dẫn local nếu đang chạy ngoài Kaggle
    local_root = Path.cwd()
    if str(local_root) not in sys.path:
        sys.path.insert(0, str(local_root))

# Định vị thư mục ảnh HyperKvasir trên Kaggle
CANDIDATE_DATA_PATHS = [
    Path("/kaggle/input/vaillant/hyperkvasir/labeled-images"),
    Path("/kaggle/input/meetnagadia/hyperkvasir/labeled-images"),
    Path("/kaggle/input/hyperkvasir/labeled-images"),
    Path("/kaggle/input/hyperkvasir-complete/labeled-images"),
    Path("/kaggle/input/hyperkvasir"),
    Path("/kaggle/input/hyperkvasir-dataset-segmented/segmented-images"),
    Path("GIEndoDL/data/raw/labeled-images"),
    Path("data/raw/labeled-images"),
    Path("D:/PhatProject/GIEndoDL/data/raw/labeled-images"),
]

DATASET_ROOT = None
for cand in CANDIDATE_DATA_PATHS:
    if cand.exists() and any(cand.iterdir()):
        DATASET_ROOT = cand
        break

if DATASET_ROOT is None:
    print("⚠️ Không tìm thấy dataset có sẵn. Đang tải tự động qua kagglehub...")
    try:
        import kagglehub
        dl_path = kagglehub.dataset_download("meetnagadia/hyperkvasir")
        DATASET_ROOT = Path(dl_path)
        print(f"✅ Tải hoàn tất qua kagglehub: {DATASET_ROOT}")
    except Exception as e:
        print(f"❌ Lỗi tải kagglehub: {e}")

print(f"📂 Thư mục ảnh sử dụng: {DATASET_ROOT}")

# Lập chỉ mục tên file ảnh sang đường dẫn tuyệt đối (0.03s)
print("⚡ Đang lập chỉ mục ảnh để truy xuất tốc độ cao...")
path_map = {}
if DATASET_ROOT is not None and DATASET_ROOT.exists():
    for root, _, files in os.walk(DATASET_ROOT):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                path_map[f] = os.path.join(root, f)
print(f"✅ Đã lập chỉ mục thành công: {len(path_map):,} file ảnh!")

In [ ]:
# 4. NẠP MANIFESTS PHÂN HOẠCH SẠCH (SCHEMA 17 CLASSES)
manifest_candidates = [
    Path("/kaggle/working/GIEndoDL/data/processed"),
    Path("GIEndoDL/data/processed"),
    Path("data/processed"),
    Path("D:/PhatProject/GIEndoDL/data/processed"),
]

processed_dir = None
for m in manifest_candidates:
    if m.exists() and (m / "train_split.csv").exists():
        processed_dir = m
        break

if processed_dir is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/processed chứa train_split.csv!")

train_df = pd.read_csv(processed_dir / "train_split.csv")
val_df = pd.read_csv(processed_dir / "val_split.csv")

CLASSES = sorted(train_df["final_class"].unique())
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

print('=' * 80)
print(f"🏷️ Số lớp mục tiêu: {NUM_CLASSES} phân lớp")
print(f"📊 Số mẫu Train:   {len(train_df):,} ảnh (70%)")
print(f"📊 Số mẫu Val:     {len(val_df):,} ảnh (15%)")
print("Danh sách 17 lớp:")
for idx, cls in enumerate(CLASSES):
    n_tr = (train_df['final_class'] == cls).sum()
    n_va = (val_df['final_class'] == cls).sum()
    print(f"  {idx:02d}. {cls:<25} -> Train: {n_tr:>4} | Val: {n_va:>4}")
print('=' * 80)

In [ ]:
# 5. CẤU HÌNH DATASET & DATALOADER VỚI POWER-LAW SAMPLER (BETA = 0.75)
from src.dataset.medical_dataloader import EndoMedicalDataset, create_power_law_weighted_sampler

TARGET_SIZE = (256, 256) # Chuẩn High-Res bảo toàn vi mạch Kudo Pit
BATCH_SIZE = 32          # Tối ưu cho GPU P100 / T4 (16GB VRAM)
NUM_WORKERS = 4          # 4 workers song song trên Kaggle Linux
BETA_SAMPLER = 0.75      # San phẳng phân bố mini-batch từ 12.6:1 xuống 6.7:1

print(f"⚙️ Khởi tạo PyTorch Medical Dataset (Target Size: {TARGET_SIZE})...")
train_dataset = EndoMedicalDataset(
    train_df, 
    str(DATASET_ROOT), 
    target_size=TARGET_SIZE, 
    is_train=True, 
    class_to_idx=class_to_idx,
    path_map=path_map
)

val_dataset = EndoMedicalDataset(
    val_df, 
    str(DATASET_ROOT), 
    target_size=TARGET_SIZE, 
    is_train=False, 
    class_to_idx=class_to_idx,
    path_map=path_map
)

# Bộ lấy mẫu có trọng số lũy thừa
sampler, class_weights = create_power_law_weighted_sampler(train_df, class_col="final_class", beta=BETA_SAMPLER)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    sampler=sampler, 
    num_workers=NUM_WORKERS, 
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

print(f"✅ Đã tạo DataLoader thành công!")
print(f"   Train Batches: {len(train_loader)} batches/epoch")
print(f"   Val Batches:   {len(val_loader)} batches/epoch")

In [ ]:
# 6. XÂY DỰNG MÔ HÌNH LAI HYBRID CNN-CBAM-TRANSFORMER (17 CLASSES)
from src.models.hybrid_cnn_cbam_transformer import build_hybrid_model

print("🧠 Khởi tạo mô hình Hybrid CNN-CBAM-Transformer...")
model = build_hybrid_model(
    num_classes=NUM_CLASSES,
    pretrained_backbone=True,
    cbam_placement="stage4",
    transformer_layers=2,
    d_model=512,
    nhead=8,
    token_pooling="gap",
    dropout=0.1
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Mô hình sẵn sàng! Tổng tham số: {total_params / 1e6:.2f} M ({trainable_params / 1e6:.2f} M trainable)")

In [ ]:
# 7. THIẾT LẬP CLASS-BALANCED FOCAL LOSS, OPTIMIZER & SCHEDULER
from src.dataset.medical_dataloader import ClassBalancedFocalLoss

samples_per_class = [int(train_df["final_class"].value_counts().get(c, 0)) for c in CLASSES]

criterion = ClassBalancedFocalLoss(
    samples_per_class=samples_per_class,
    num_classes=NUM_CLASSES,
    beta=0.75,
    gamma=2.0,
    label_smoothing=0.05
).to(device)

EPOCHS = 10
BASE_LR = 1e-4
WEIGHT_DECAY = 1e-2

optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print(f"🎯 Loss Function: Class-Balanced Focal Loss (beta=0.75, gamma=2.0, smoothing=0.05)")
print(f"⚙️ Optimizer:     AdamW (lr={BASE_LR}, weight_decay={WEIGHT_DECAY})")
print(f"📈 Scheduler:     CosineAnnealingLR (T_max={EPOCHS}, eta_min=1e-6)")
print(f"⚡ AMP FP16:      {device.type == 'cuda'}")

In [ ]:
# 8. THỰC THI DRY-RUN HUẤN LUYỆN 10 EPOCHS
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

output_dir = Path("/kaggle/working/kaggle_dryrun_d46_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

history = []
per_class_f1_records = []
best_val_macro_f1 = 0.0
t_start_all = time.time()

print('=' * 90)
print(f"🚀 BẮT ĐẦU PHIÊN HUẤN LUYỆN DRY-RUN {EPOCHS} EPOCHS TRÊN KAGGLE GPU")
print('=' * 90)

for epoch in range(1, EPOCHS + 1):
    t_epoch_0 = time.time()
    cur_lr = optimizer.param_groups[0]['lr']
    
    # --- A. TRAIN PHASE ---
    model.train()
    train_loss = 0.0
    train_preds, train_targets = [], []
    t_train_0 = time.time()
    
    pbar = tqdm(train_loader, desc=f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train", leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["label"].to(device, non_blocking=True)
        b_size = images.size(0)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(dtype=torch.float16, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss = criterion(logits, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * b_size
        train_preds.extend(torch.argmax(logits, dim=-1).detach().cpu().numpy())
        train_targets.extend(targets.detach().cpu().numpy())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    train_fps = len(train_dataset) / max(time.time() - t_train_0, 1e-5)
    train_loss = train_loss / len(train_dataset)
    train_acc = accuracy_score(train_targets, train_preds)
    
    # --- B. VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []
    t_val_0 = time.time()
    
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["label"].to(device, non_blocking=True)
            b_size = images.size(0)
            with torch.cuda.amp.autocast(dtype=torch.float16, enabled=(device.type == 'cuda')):
                logits = model(images)
                loss = criterion(logits, targets)
            val_loss += loss.item() * b_size
            val_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            val_targets.extend(targets.cpu().numpy())
            
    val_fps = len(val_dataset) / max(time.time() - t_val_0, 1e-5)
    val_loss = val_loss / len(val_dataset)
    val_acc = accuracy_score(val_targets, val_preds)
    val_bal_acc = balanced_accuracy_score(val_targets, val_preds)
    val_macro_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
    val_weighted_f1 = f1_score(val_targets, val_preds, average="weighted", zero_division=0)
    
    # F1 score từng phân lớp
    f1_per_c = f1_score(val_targets, val_preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    f1_dict = {CLASSES[i]: float(f1_per_c[i]) for i in range(NUM_CLASSES)}
    f1_record = {"epoch": epoch}
    f1_record.update(f1_dict)
    per_class_f1_records.append(f1_record)
    
    scheduler.step()
    epoch_time = time.time() - t_epoch_0
    
    is_best = val_macro_f1 > best_val_macro_f1
    if is_best:
        best_val_macro_f1 = val_macro_f1
        torch.save(model.state_dict(), output_dir / "best_dryrun_model.pth")
        
    history.append({
        "epoch": epoch, "lr": cur_lr,
        "train_loss": train_loss, "train_acc": train_acc, "train_fps": train_fps,
        "val_loss": val_loss, "val_acc": val_acc, "val_bal_acc": val_bal_acc,
        "val_macro_f1": val_macro_f1, "val_weighted_f1": val_weighted_f1, "val_fps": val_fps,
        "epoch_time_s": epoch_time
    })
    
    star = " ★ BEST" if is_best else ""
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
          f"Tr Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Acc: {val_acc*100:.2f}% | "
          f"Val Macro F1: {val_macro_f1*100:.2f}% | "
          f"Train FPS: {train_fps:.1f} | Val FPS: {val_fps:.1f} | "
          f"Time: {epoch_time:.1f}s{star}")

total_sec = time.time() - t_start_all
print('=' * 90)
print(f"✅ ĐÃ HOÀN TẤT DRY-RUN 10 EPOCHS TRONG {total_sec:.1f} GIÂY ({total_sec/60:.2f} PHÚT)!")
print(f"🏆 BEST VAL MACRO F1: {best_val_macro_f1 * 100:.2f}%")
print('=' * 90)

In [ ]:
# 9. TRỰC QUAN HÓA TIẾN TRÌNH HỘI TỤ & HIỆU NĂNG TỪNG LỚP
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
hist_df.to_csv(output_dir / "dryrun_metrics_history.csv", index=False)

f1_df = pd.DataFrame(per_class_f1_records)
f1_df.to_csv(output_dir / "dryrun_per_class_f1.csv", index=False)

# Biểu đồ 4 đồ thị con
fig, axes = plt.subplots(2, 2, figsize=(15, 10), dpi=300)
ep = hist_df["epoch"]

# 1. Loss
axes[0, 0].plot(ep, hist_df["train_loss"], "o-", color="#1f77b4", label="Train Loss", lw=2)
axes[0, 0].plot(ep, hist_df["val_loss"], "s--", color="#d62728", label="Val Loss", lw=2)
axes[0, 0].set_title("Biến thiên Hàm Mất mát (Class-Balanced Focal Loss)", fontweight="bold")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle="--", alpha=0.5)

# 2. Macro F1 & Balanced Acc
axes[0, 1].plot(ep, hist_df["val_macro_f1"] * 100, "o-", color="#2ca02c", label="Val Macro F1 (%)", lw=2)
axes[0, 1].plot(ep, hist_df["val_bal_acc"] * 100, "^--", color="#9467bd", label="Val Balanced Acc (%)", lw=2)
axes[0, 1].set_title("Chỉ số Đánh giá Cốt lõi (Macro F1 & Balanced Acc)", fontweight="bold")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Phần trăm (%)")
axes[0, 1].legend(loc="lower right")
axes[0, 1].grid(True, linestyle="--", alpha=0.5)

# 3. Accuracy & Overfit Gap
axes[1, 0].plot(ep, hist_df["train_acc"] * 100, "o-", color="#1f77b4", label="Train Acc (%)", lw=2)
axes[1, 0].plot(ep, hist_df["val_acc"] * 100, "s--", color="#ff7f0e", label="Val Acc (%)", lw=2)
axes[1, 0].fill_between(ep, hist_df["train_acc"] * 100, hist_df["val_acc"] * 100, color="gray", alpha=0.15, label="Overfit Gap")
axes[1, 0].set_title("Độ chính xác Top-1 & Khoảng cách Quá khớp", fontweight="bold")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Accuracy (%)")
axes[1, 0].legend(loc="lower right")
axes[1, 0].grid(True, linestyle="--", alpha=0.5)

# 4. Throughput (FPS)
axes[1, 1].plot(ep, hist_df["train_fps"], "o-", color="#17becf", label="Train Throughput (FPS)", lw=2)
axes[1, 1].plot(ep, hist_df["val_fps"], "s--", color="#bcbd22", label="Val Throughput (FPS)", lw=2)
axes[1, 1].axhline(y=hist_df["train_fps"].mean(), color="blue", linestyle=":", label=f"Mean Train FPS: {hist_df['train_fps'].mean():.1f}")
axes[1, 1].set_title("Thông lượng Xử lý Dữ liệu (Throughput / FPS)", fontweight="bold")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("FPS")
axes[1, 1].legend(loc="lower right")
axes[1, 1].grid(True, linestyle="--", alpha=0.5)

plt.suptitle("GIEndoDL: KẾT QUẢ HUẤN LUYỆN THỬ NGHIỆM DRY-RUN 10 EPOCHS (TASK D46)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "dryrun_convergence_curves.png", dpi=300, bbox_inches="tight")
plt.show()

# Biểu đồ Bar Chart Per-Class F1
final_f1 = f1_df.iloc[-1].drop("epoch").sort_values(ascending=False)
plt.figure(figsize=(12, 7), dpi=300)
colors = ["#2ca02c" if v >= 0.80 else "#ff7f0e" if v >= 0.65 else "#d62728" for v in final_f1.values]
bars = plt.barh(final_f1.index, final_f1.values * 100, color=colors, edgecolor="black", alpha=0.85)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 1.0, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va="center", fontsize=9, fontweight="bold")
plt.axvline(x=final_f1.mean() * 100, color="navy", linestyle="--", lw=2, label=f"Mean Macro F1: {final_f1.mean()*100:.2f}%")
plt.title("F1-Score trên 17 Phân lớp Bệnh học tại Epoch 10", fontsize=13, fontweight="bold")
plt.xlabel("F1-Score (%)")
plt.xlim(0, 105)
plt.legend(loc="lower right")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(output_dir / "dryrun_per_class_f1_barchart.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 10. ĐÓNG GÓI KẾT QUẢ & TẠO LINK TẢI 1-CLICK TRÊN KAGGLE
import shutil
from IPython.display import FileLink, display, HTML

zip_path = "/kaggle/working/kaggle_dryrun_d46_results"
shutil.make_archive(zip_path, "zip", output_dir)
full_zip_file = f"{zip_path}.zip"

print('=' * 80)
print(f"📦 Đã đóng gói thành công: {full_zip_file} ({os.path.getsize(full_zip_file) / (1024**2):.2f} MB)")
print("📥 Bấm vào liên kết bên dưới để tải toàn bộ kết quả về máy tính:")
display(FileLink("kaggle_dryrun_d46_results.zip"))
print('=' * 80)